# M4 · The training loop, from scratch

In M2 you ran a model that someone else had trained. In this module you train one yourself, from random numbers to something that writes like Shakespeare.

"Training" can sound mysterious. It isn't. It's one short loop, repeated a few thousand times. By the end of this lesson you'll have seen every line of it, and in the exercises you'll write it yourself. In M5 you'll run the same loop on a real pretrained model.

By the end you'll have:

- turned a text into training examples
- measured how wrong a model's guesses are (the loss)
- run training steps by hand and watched the loss fall
- used the three tricks every real training run uses: a learning-rate schedule, gradient clipping and gradient accumulation
- read a loss curve, and seen what overfitting looks like

**How to work through it** (same as M1 to M3)

- Run each cell from top to bottom with **Shift-Enter**, and read the **What each line does** box under it.
- When a cell says **Predict**, write your guess down before running it.
- New words are **bold** where they're explained, and collected in the glossary in `README.md`.
- Save with **Cmd-S** now and then so the dashboard sees your progress.

Nothing to download: the text is in this folder. A few cells train small models, and take up to a minute.

## 0. What training really does

In M1 you trained a small network in four beats, over and over:

1. make a guess with the current weights
2. measure how wrong it was: the **loss**
3. work out which way to nudge each weight to make the loss smaller: the gradients
4. nudge every weight a little in that direction

**Training** is just those four beats repeated. Each repeat is one **step**.

For a language model the guess is "which character comes next?" The model in `model.py` is a tiny GPT, built from the same parts you met in M2: embeddings, causal attention, an MLP. It reads characters and scores every possible next character. It's given to you; this module is about training it.

Here it is before any training, with random weights:

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from model import TinyGPT, count_parameters

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)

text = open("shakespeare.txt").read()
chars = sorted(set(text))
untrained = TinyGPT(vocab_size=len(chars))
print(f"{count_parameters(untrained):,} parameters, on {device}")

out = untrained.generate(torch.tensor([[0]]), 80)[0].tolist()
print(repr("".join(chars[i] for i in out)))

**What each line does**

- `device`: `"mps"` is your Mac's GPU (M0). The small models in this lesson run there when it's available.
- `torch.manual_seed(0)`: fixes the random numbers, so you get the same results as these notes.
- `open("shakespeare.txt").read()`: the whole training text as one string. It's about 40 of Shakespeare's plays, 1.1 million characters.
- `sorted(set(text))`: every different character in the text, in order. These are the model's tokens.
- `TinyGPT(vocab_size=len(chars))`: a new model with random weights. `count_parameters` adds up how many numbers it has: about 4.8 million (SmolLM2 in M2 had 135 million).
- `untrained.generate(...)`: writes 80 characters, one at a time, like your M2 loop. With random weights it's gibberish: every character is about equally likely.

By the end of this module, the same model will write something like this:

```
First Senator:
I will live carry my life, and bound I have,
For then strew-struction the happy offence,
```

Not real Shakespeare, but real words, names in capitals, and lines that scan. All it learned from is the text, one guessed character at a time.

## 1. Characters as tokens, and the train/validation split

In M3 you built a tokenizer that merges bytes into bigger tokens. Here we keep it as simple as possible: **one character is one token**. That gives a vocabulary of 65, so the model only ever has 65 choices.

In [ ]:
print(len(text), "characters,", len(chars), "different ones")
print(repr("".join(chars)))

stoi = {c: i for i, c in enumerate(chars)}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(chars[i] for i in ids)


print(encode("Hi!"))
print(decode(encode("Hi!")))

**What each line does**

- The 65 characters are a newline, a space, some punctuation, the digit 3, and the capital and small letters.
- `stoi`: "string to integer", a dict from each character to its id. `enumerate(chars)` numbers them 0, 1, 2, ...
- `encode` and `decode`: a one-character tokenizer. `encode` looks each character up in `stoi`; `decode` looks each id up in `chars` and joins them back into a string.

Now the whole text becomes one long tensor of ids. Then we split it in two:

- the **training set**: the first 90%. The model learns from this.
- the **validation set**: the last 10%. The model never trains on it. We only use it to measure how well the model does on text it hasn't seen.

Why bother? A model can get very good at the exact text it trains on, by memorising it, without learning anything that carries over to new text. The validation set is the honest exam: the questions aren't in the textbook.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

print(data.shape, train_data.shape, val_data.shape)
print(decode(val_data[:60].tolist()))

**What each line does**

- `torch.tensor(..., dtype=torch.long)`: the ids as a tensor of whole numbers. Embedding layers want `long` ids.
- `int(0.9 * len(data))`: where the 90% mark is. `data[:split]` is everything before it, `data[split:]` everything after.
- The validation set starts partway through a play the model will never train on.

## 2. Batches of shifted windows

The model doesn't read the whole text at once. It reads a window of characters at a time. The window's length is the **context length** (also called the block size): how many characters the model can see when it makes a guess. Ours is 256 in the real run; we'll use 8 here so you can read it.

One window gives many examples at once. Take 9 characters in a row: the input `x` is the first 8, and the target `y` is the same 8 moved along by one.

**Predict:** if `x` is `First Ci`, what is `y`?

In [ ]:
block_size = 8
i = 0
x = train_data[i : i + block_size]
y = train_data[i + 1 : i + block_size + 1]

print(repr(decode(x.tolist())), "->", repr(decode(y.tolist())))
for t in range(block_size):
    print(f"  after {decode(x[: t + 1].tolist())!r:12} comes {decode([y[t].item()])!r}")

**What each line does**

- `train_data[i : i + block_size]`: 8 ids in a row, starting at position `i`. That's `x`.
- `train_data[i + 1 : i + block_size + 1]`: the same window, one further along. That's `y`: `irst Cit`.
- The loop shows what that one window teaches. `y[t]` is the character after `x[0]` to `x[t]`: after `F` comes `i`, after `Fi` comes `r`, and so on. So one window of 8 holds 8 guessing exercises.
- This is why the attention in M2 had to be causal: when guessing `y[t]`, the model may only look at `x[0]` to `x[t]`, never at the answer further along.

A **batch** is several windows trained on together, each from a random place in the text. Random places matter: if the windows always came in order, the model would see a whole play, then the next, and forget the first.

`torch.randint` picks random starts, and `torch.stack` piles the windows into one tensor.

**Predict:** what shape is `xb`?

In [ ]:
g = torch.Generator().manual_seed(0)
starts = torch.randint(len(train_data) - block_size, (4,), generator=g)
xb = torch.stack([train_data[s : s + block_size] for s in starts])
yb = torch.stack([train_data[s + 1 : s + block_size + 1] for s in starts])

print(starts)
print(xb.shape, yb.shape)
for row_x, row_y in zip(xb, yb):
    print(repr(decode(row_x.tolist())), "->", repr(decode(row_y.tolist())))

**What each line does**

- `torch.Generator().manual_seed(0)`: a random-number source of our own, so the same seed always picks the same starts.
- `torch.randint(high, (4,), generator=g)`: 4 random whole numbers from 0 up to (not including) `high`. `high` is `len(train_data) - block_size` so that even the last start leaves room for `y`'s extra character.
- `torch.stack([...])`: turns a list of four tensors of shape `(8,)` into one tensor of shape `(4, 8)`: one row per window.
- `xb` and `yb` are both `(4, 8)`: batch size by context length. That's exercise 1.

## 3. The causal LM loss

Give the model a batch and it returns **logits**: one score for every possible next character, at every position. So for a `(4, 8)` batch of ids, it returns `(4, 8, 65)`.

To turn scores into a loss we use **cross-entropy**, the same loss as M1's classifier. Softmax (M2) turns the 65 scores into probabilities, and the loss is `-ln(probability of the right character)`:

- the model gave the right character probability 1: loss is `-ln 1 = 0`
- probability 0.5: loss is about 0.69
- probability 1/65, an even guess: loss is `ln 65`, about 4.17

Averaged over every position in the batch, this is the **causal LM loss** (causal language-modelling loss): how surprised the model is, on average, by each next character.

In [ ]:
small = TinyGPT(vocab_size=65, block_size=8, n_embd=32, n_head=2, n_layer=1)
logits = small(xb)
print(logits.shape)

batch, time, vocab = logits.shape
flat_logits = logits.reshape(batch * time, vocab)
flat_targets = yb.reshape(batch * time)
print(flat_logits.shape, flat_targets.shape)

loss = F.cross_entropy(flat_logits, flat_targets)
print(loss.item(), math.log(65))

**What each line does**

- `small(xb)`: a small model's guesses for the batch: shape `(4, 8, 65)`.
- `batch, time, vocab = logits.shape`: unpacks the three sizes into three names: 4, 8 and 65.
- `F.cross_entropy` wants one row of scores per guess, shaped `(guesses, vocab)`, and one right answer per guess. `.reshape(batch * time, vocab)` lays the 4 x 8 positions out as 32 rows; `yb.reshape(batch * time)` does the same to the answers.
- `loss.item()`: about 4.6, a little above `math.log(65)` (the natural log, 4.17). Random weights make guesses that are nearly even, but not exactly. That's exercise 2.
- Always check your first loss against `ln(vocab size)`. Close to it is right. If it starts far away (10, or 50), something is wrong before training even begins.

## 4. One training step

Here are the four beats from section 0 as PyTorch code. The **optimizer** is the part that nudges the weights. We use **AdamW**, the optimizer almost every language model is trained with. It's a smarter version of M1's SGD: it keeps a running average of each weight's recent gradients and scales each nudge to fit.

How big each nudge is depends on the **learning rate**.

```python
optimizer.zero_grad()           # clear the gradients from the last step
loss = F.cross_entropy(...)     # 1-2. guess, and measure the loss
loss.backward()                 # 3. work out every weight's gradient
optimizer.step()                # 4. nudge every weight
```

Let's run 200 steps on a small model and watch the loss.

In [ ]:
def get_batch(data, block_size, batch_size, generator):
    starts = torch.randint(len(data) - block_size, (batch_size,), generator=generator)
    x = torch.stack([data[s : s + block_size] for s in starts])
    y = torch.stack([data[s + 1 : s + block_size + 1] for s in starts])
    return x.to(device), y.to(device)


def make_model():
    torch.manual_seed(0)
    model = TinyGPT(vocab_size=65, block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.0)
    return model.to(device)


def train_for(model, steps, lr, data=train_data, report=50):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(1)
    losses = []
    for step in range(steps):
        x, y = get_batch(data, 64, 16, g)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, 65), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if report and (step + 1) % report == 0:
            print(f"  step {step + 1:4d}  loss {loss.item():.3f}")
    return losses


model = make_model()
losses = train_for(model, 200, lr=3e-3)
print(repr(decode(model.generate(torch.tensor([[0]], device=device), 120)[0].tolist())))

**What each line does**

- `get_batch`: section 2's batch, wrapped in a function. `.to(device)` moves the batch to the GPU, where the model is.
- `make_model`: a small model (about 100,000 parameters) so these cells run in seconds. Seeding first means every call makes the same starting model. `dropout=0.0` switches off dropout, which section 8 explains, so the comparisons below are exact.
- `torch.optim.AdamW(model.parameters(), lr=lr)`: an optimizer in charge of every weight in the model. `lr` is the learning rate.
- `logits.reshape(-1, 65)`: `-1` means "whatever size makes it fit", here 16 x 64 = 1024 rows.
- The four beats: `optimizer.zero_grad()`, the loss, `loss.backward()`, `optimizer.step()`. Gradients add up on every `.backward()` unless you clear them; you'll use that on purpose in section 7.
- `losses.append(loss.item())`: keeps each step's loss as a plain number, for plotting.
- The loss falls from about 4.6 to about 2.4 in 200 steps, and the sample already has spaces, short words and line breaks.

## 5. Learning-rate schedules

The learning rate is the most important number in training. Let's train the same model three times with three learning rates.

**Predict:** which one ends with the lowest loss?

In [ ]:
runs = {}
for lr in [3e-4, 3e-3, 1e-1]:
    print(f"lr {lr}:")
    runs[lr] = train_for(make_model(), 200, lr=lr, report=100)

for lr, curve in runs.items():
    plt.plot(curve, label=f"lr {lr}")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

**What each line does**

- Each run starts from the same model (`make_model()` seeds first), so only the learning rate differs.
- `plt.plot(curve, ...)`: draws one run's loss at every step. `plt.legend()` adds the labels.
- `3e-4` (0.0003) learns, but slowly: its steps are too small.
- `1e-1` (0.1) is far too big. Each step jumps right past the low point and lands somewhere worse, so the loss bounces and gets stuck high. With some models it shoots up to `nan` ("not a number") and the run is ruined.
- `3e-3` is the best of the three here. The right value depends on the model and batch size, so people try a few.

Real training runs don't keep the learning rate fixed. They follow a **learning-rate schedule** with two parts:

- **Warmup**: start small and climb in a straight line to the peak over the first few hundred steps. At the start the weights are random and AdamW's running averages are empty, so the first gradients are big and unreliable. Small first steps keep one bad update from wrecking the model.
- **Cosine decay**: then glide down to a small minimum along half a cosine wave. Big steps early cover ground fast; small steps late settle into a good spot instead of bouncing around it.

The cosine part uses `math.cos`. As `progress` goes from 0 to 1, `(1 + math.cos(math.pi * progress)) / 2` slides smoothly from 1 to 0:

In [ ]:
max_lr, min_lr, warmup_steps, total_steps = 1e-3, 1e-4, 100, 2000


def warmup(step):
    return max_lr * (step + 1) / warmup_steps


def cosine(step):
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


print(warmup(0), warmup(49), warmup(99))
print(cosine(100), cosine(1050), cosine(2000))

steps = range(total_steps)
plt.plot(steps, [warmup(s) if s < warmup_steps else cosine(s) for s in steps])
plt.xlabel("step")
plt.ylabel("learning rate")
plt.show()

**What each line does**

- `warmup(step)`: climbs in a straight line. Step 0 gets 1/100 of `max_lr`, step 99 gets all of it. The `+ 1` means step 0 doesn't start at exactly 0, which would waste a step.
- `cosine(step)`: `progress` is 0 right after warmup and 1 at `total_steps`. `math.cos(math.pi * progress)` goes from 1 down to -1, so the rate goes from `max_lr` down to `min_lr`.
- `cosine(1050)` is halfway: exactly halfway between `max_lr` and `min_lr`, 0.00055.
- `warmup(s) if s < warmup_steps else cosine(s)`: a conditional expression, picking which part applies at each step.
- Exercise 3 puts these together into `lr_at`, and adds one more case: from `total_steps` on, stay at `min_lr`.

## 6. Gradient clipping

Now and then one batch produces huge gradients: an odd stretch of text, or just bad luck. One giant step can undo hours of training.

**Gradient clipping** puts a limit on how big a step can get. First measure all the gradients together, as if every gradient value of every weight were one long list, and take its length. That length is the **gradient norm**: the square root of the sum of every value squared.

In [ ]:
model = make_model()
x, y = get_batch(train_data, 64, 16, torch.Generator().manual_seed(0))
loss = F.cross_entropy(model(x).reshape(-1, 65), y.reshape(-1))
loss.backward()

grads = [p.grad for p in model.parameters() if p.grad is not None]
total = torch.sqrt(sum((g**2).sum() for g in grads)).item()
print(len(grads), "gradient tensors, total norm", round(total, 3))

print(torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1e9).item())

**What each line does**

- `p.grad`: after `.backward()`, every weight tensor `p` has a `.grad` of the same shape (M1).
- `if p.grad is not None`: a weight that wasn't used has no gradient (`None`). The list comprehension skips those.
- `(g**2).sum()`: one tensor's values squared and added up. `sum(...)` adds those up across every tensor; `torch.sqrt` takes the square root. `.item()` makes it a plain number.
- `torch.nn.utils.clip_grad_norm_`: PyTorch's built-in version. It returns the same norm. (`max_norm=1e9` is so large that nothing gets clipped.) In exercise 4 you write your own.

If the norm is above a limit, `max_norm`, scale every gradient down by the same factor: `max_norm / total`. Everything shrinks together, so the step still points the same way; it's just shorter. If the norm is already below the limit, leave it alone: clipping only ever shrinks.

**Predict:** a gradient `[3, 4]` has norm 5. Clipped to `max_norm = 1`, what does it become?

In [ ]:
g = torch.tensor([3.0, 4.0])
norm = torch.sqrt((g**2).sum()).item()
max_norm = 1.0
print("before", g, "norm", norm, "direction", g / norm)

if norm > max_norm:
    g.mul_(max_norm / norm)
print("after ", g, "norm", torch.sqrt((g**2).sum()).item(), "direction", g / max_norm)

**What each line does**

- `norm`: `sqrt(3² + 4²) = 5`.
- `g / norm`: the direction alone, `[0.6, 0.8]`.
- `g.mul_(max_norm / norm)`: multiplies `g` by 1/5 *in place*. The `_` at the end of a PyTorch method means it changes the tensor itself instead of making a new one. That matters here: the optimizer reads `p.grad`, so you have to change that tensor, not a copy.
- After: `[0.6, 0.8]`, norm 1, and the direction hasn't changed.

## 7. Gradient accumulation

Bigger batches give steadier gradients, because they average over more examples. But a bigger batch needs more memory, and on a laptop memory runs out fast (you'll feel this in M5).

**Gradient accumulation** gets the effect of a big batch with the memory of a small one. Split the big batch into smaller pieces, each called a **micro-batch**. Run `.backward()` on each one without clearing the gradients in between: PyTorch *adds* each new gradient to `.grad`. Then take one optimizer step.

One detail: the loss is an *average*. The loss of a batch of 16 is the average over 16 windows; the sum of two half-batch losses would be twice too big. So divide each micro-batch's loss by the number of micro-batches before `.backward()`.

Let's check that it really is the same:

In [ ]:
x, y = get_batch(train_data, 64, 16, torch.Generator().manual_seed(0))

big = make_model()
F.cross_entropy(big(x).reshape(-1, 65), y.reshape(-1)).backward()

acc = make_model()
micro_batches = [(x[:8], y[:8]), (x[8:], y[8:])]
for xm, ym in micro_batches:
    loss = F.cross_entropy(acc(xm).reshape(-1, 65), ym.reshape(-1)) / len(micro_batches)
    loss.backward()

same = all(
    torch.allclose(a.grad, b.grad, atol=1e-6) for a, b in zip(big.parameters(), acc.parameters())
)
print("same gradients:", same)

**What each line does**

- `big`: one batch of 16, one `.backward()`.
- `acc`: the same 16 windows as two micro-batches of 8. Each loss is divided by `len(micro_batches)`, 2, and each `.backward()` adds to the gradients.
- `zip(big.parameters(), acc.parameters())`: pairs up the matching weights of the two models. `torch.allclose` checks the gradients are equal, allowing for tiny rounding differences.
- `same gradients: True`. Try deleting `/ len(micro_batches)`: the gradients come out twice as big.

The **effective batch size** is the micro-batch size times the number of micro-batches: how many windows each optimizer step learns from. `train.py` uses 2 micro-batches of 16, so 32.

Exercise 5, `train_step`, is one full step with all of this: clear the gradients once, accumulate over the micro-batches, clip, then step.

## 8. Reading loss curves

The loss printed during training comes from one batch at a time, so it's noisy. To get an honest measurement, average the loss over several batches, and do it carefully:

- **`model.eval()`** switches the model to evaluation mode. The main thing it changes is **dropout**: during training, dropout switches off a random 10% of the model's numbers at every step, so the model can't lean too hard on any one of them. That helps it learn, but makes measurements noisy. `model.eval()` turns it off; `model.train()` turns it back on.
- **`torch.no_grad()`** (M1): we're only measuring, so there's no need to record anything for gradients. It's faster and uses less memory.

In [ ]:
def estimate(model, data, iters=10):
    g = torch.Generator().manual_seed(2)
    model.eval()
    total = 0.0
    with torch.no_grad():
        for _ in range(iters):
            x, y = get_batch(data, 64, 16, g)
            total += F.cross_entropy(model(x).reshape(-1, 65), y.reshape(-1)).item()
    model.train()
    return total / iters


model = make_model()
train_for(model, 300, lr=3e-3, report=0)
print("train", round(estimate(model, train_data), 3), " val", round(estimate(model, val_data), 3))
print("back in training mode:", model.training)

**What each line does**

- `model.eval()` ... `model.train()`: evaluation mode while measuring, then back to training mode. Forget `model.train()` and dropout stays off for the rest of training.
- `with torch.no_grad():` the indented block runs without recording gradients.
- `total / iters`: the average over 10 batches, much steadier than one.
- `model.training`: `True` in training mode, `False` in evaluation mode.
- After 300 steps, the train and validation losses are close: the model is learning patterns that carry over to text it hasn't seen. That's exercise 6.

A **loss curve** is the loss plotted against the step, for both sets. Most of what you need to know about a run is in it.

What does it look like when things go wrong? Let's force the classic problem: train on a tiny slice of the text, 5,000 characters, for a long time.

**Predict:** what will the validation loss do?

In [ ]:
tiny_train = train_data[:5000]
model = make_model()
points = [(0, estimate(model, tiny_train), estimate(model, val_data))]
for round_ in range(8):
    train_for(model, 100, lr=3e-3, data=tiny_train, report=0)
    points.append(((round_ + 1) * 100, estimate(model, tiny_train), estimate(model, val_data)))
    print(f"  step {points[-1][0]}  train {points[-1][1]:.3f}  val {points[-1][2]:.3f}")

plt.plot([p[0] for p in points], [p[1] for p in points], label="train (5,000 characters)")
plt.plot([p[0] for p in points], [p[2] for p in points], label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

**What each line does**

- `train_data[:5000]`: a slice of 5,000 characters, a page or two of one play.
- The loop trains for 100 steps at a time (on the slice only) and measures both losses after each round. `round_` has an underscore because `round` is already a Python function.
- `points`: starts with the losses before any training, at step 0. Each point is (step, train loss, validation loss); `points[-1]` is the last one.
- The training loss keeps falling, towards 0: the model is memorising the page. The validation loss falls at first (from about 4.6 to under 3), then turns and climbs steadily: what it memorises doesn't carry over to other text.

That's **overfitting**: the model gets better at its training text and worse at everything else. On a loss curve you see the two lines split, the training loss still falling while the validation loss climbs.

What to do about it: stop around where the validation loss was lowest, give the model more data, or make it harder to memorise (more dropout, a smaller model). With 1 million characters and 2,000 steps, the real run sees each character only a few times, so it barely overfits. You'll see a small gap open up near the end of your loss curve.

The validation loss is the number that matters. It's the one the checkpoint checks.

## 9. Your task

You've now seen every piece. Time to write them yourself.

1. Open `exercises.py`. There are six exercises, each with plain words and numbered steps, and each points to the section above that shows what you need:
   1. `get_batch` (section 2)
   2. `causal_lm_loss` (section 3)
   3. `lr_at` (section 5)
   4. `clip_grad_norm` (section 6)
   5. `train_step` (sections 4 and 7)
   6. `estimate_loss` (section 8)
2. Check your work as you go: `make check M=04`. The run tests fail until step 4.
3. Answer the six questions in `answers.toml`. Predict first, then check yourself in this notebook.
4. Run `uv run python modules/m04_training_loop/train.py`. It trains the full 4.8M-parameter model on your Mac's GPU with your six functions: 2,000 steps, about 2 minutes. It prints the losses as it goes, then a sample, and saves `outputs/m04/loss_curve.png` and `samples.txt`. It needs a validation loss below 1.70, and 80% of the words in the sample must be real words from the training text.
5. Run `make check M=04` again. When everything passes, M4 turns green on the dashboard.

Stuck? `make hint M=04 EX=5` gives a nudge for exercise 5. Every term is in the glossary in `README.md`.

**Stretch:** run `train.py --precision bf16`. It does the maths in bfloat16, a 16-bit number format (M0), instead of 32-bit. Compare the time, the peak memory and the final loss with your first run.